# 🏆 Mora SP Cup 2026 — Team Zentrix: **v10 APEX — FULL POWER**
## 200-Epoch Fresh Start | All 460 Images | Progressive Loss Curriculum | CosineWarmRestarts × 4
### ✅ Kaggle + Colab Compatible (No magic commands)

### 🔥 Why v10 APEX Will Break Through 0.63+:
1. **🆕 Fresh Random Initialization**: v8/v9 were stuck in v7's local minimum. Fresh weights → new loss landscape.
2. **📦 Full 460-Image Training (No Holdout)**: Competition test set is 20 *different* images. Train on ALL 460.
3. **📈 Progressive Loss Curriculum**:
   - **Phase 1 (Ep 1–100)**: Pure Charbonnier — Max PSNR optimization
   - **Phase 2 (Ep 101–160)**: + 0.05×MS-FFT — Texture recovery
   - **Phase 3 (Ep 161–200)**: + 0.03×Soft-SSIM — Structure lock-in
4. **🔄 CosineAnnealingWarmRestarts (T₀=50)**: 4 cycles — escapes local minima.
5. **⚡ EMA (β=0.999) + SWA (Final 20 Epochs)**
6. **⏱ ~160–180 min total** on T4 GPU.

### 🚀 Quick Start (Kaggle):
1. Upload your `competition_data/public/` folder as a Kaggle dataset.
2. Set Accelerator → **GPU T4 x2** in Settings.
3. Run all cells!

### 🚀 Quick Start (Colab):
1. Runtime → Disconnect & delete runtime.
2. Runtime → Run all.
3. `best_denoiser_v10.pth` auto-downloads!


In [ ]:
# [1] Verify GPU & Install
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    props = torch.cuda.get_device_properties(0)
    print(f'\u2705 GPU: {props.name} | VRAM: {props.total_memory/1024**3:.1f} GB')
else:
    raise RuntimeError('\u274c No GPU! Enable GPU in Settings.')
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.free,memory.total', '--format=csv'], check=False)
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless', 'scikit-image', 'PyWavelets'], check=False)
print('\u2705 Packages ready!')

In [ ]:
# [2] Create Directory Structure
import os
for d in ['scripts/checkpoints', 'models',
          'competition_data/public/noisy',
          'competition_data/public/ground_truth']:
    os.makedirs(d, exist_ok=True)
print('\u2705 Directories ready!')

In [ ]:
# [3] Write NAFNet Architecture — scripts/models.py (Kaggle-compatible)
import os
os.makedirs('scripts', exist_ok=True)

models_code = '''
import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass SimpleGate(nn.Module):\n    def forward(self, x):\n        x1, x2 = x.chunk(2, dim=1)\n        return x1 * x2\n\nclass NAFBlock(nn.Module):\n    def __init__(self, c, DW_Expand=2, FFN_Expand=2, drop_out_rate=0.0):\n        super().__init__()\n        dw = c * DW_Expand\n        self.conv1 = nn.Conv2d(c, dw, 1, bias=True)\n        self.conv2 = nn.Conv2d(dw, dw, 3, 1, 1, groups=dw, bias=True)\n        self.conv3 = nn.Conv2d(dw//2, c, 1, bias=True)\n        self.sca = nn.Sequential(nn.AdaptiveAvgPool2d(1),\n                                  nn.Conv2d(dw//2, dw//2, 1, bias=True))\n        self.sg = SimpleGate()\n        ffn = FFN_Expand * c\n        self.conv4 = nn.Conv2d(c, ffn, 1, bias=True)\n        self.conv5 = nn.Conv2d(ffn//2, c, 1, bias=True)\n        self.norm1 = nn.GroupNorm(1, c)\n        self.norm2 = nn.GroupNorm(1, c)\n        self.dropout1 = nn.Dropout(drop_out_rate) if drop_out_rate > 0 else nn.Identity()\n        self.dropout2 = nn.Dropout(drop_out_rate) if drop_out_rate > 0 else nn.Identity()\n        self.beta  = nn.Parameter(torch.zeros(1, c, 1, 1))\n        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))\n\n    def forward(self, inp):\n        x = self.norm1(inp)\n        x = self.conv1(x)\n        x = self.conv2(x)\n        x = self.sg(x)\n        x = x * self.sca(x)\n        x = self.conv3(x)\n        x = self.dropout1(x)\n        y = inp + x * self.beta\n        x = self.norm2(y)\n        x = self.conv4(x)\n        x = self.sg(x)\n        x = self.conv5(x)\n        x = self.dropout2(x)\n        return y + x * self.gamma\n\nclass NAFNet(nn.Module):\n    def __init__(self, img_channel=3, width=64,\n                 enc_blks=(2, 2, 4, 8), dec_blks=(2, 2, 2, 2), middle_blk_num=8):\n        super().__init__()\n        self.intro  = nn.Conv2d(img_channel, width, 3, 1, 1, bias=True)\n        self.ending = nn.Conv2d(width, img_channel, 3, 1, 1, bias=True)\n        self.encoders, self.decoders = nn.ModuleList(), nn.ModuleList()\n        self.ups, self.downs = nn.ModuleList(), nn.ModuleList()\n        chan = width\n        for num in enc_blks:\n            self.encoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))\n            self.downs.append(nn.Conv2d(chan, chan*2, 2, 2))\n            chan *= 2\n        self.middle_blks = nn.Sequential(*[NAFBlock(chan) for _ in range(middle_blk_num)])\n        for num in dec_blks:\n            self.ups.append(nn.Sequential(\n                nn.Conv2d(chan, chan*2, 1, bias=False), nn.PixelShuffle(2)))\n            chan //= 2\n            self.decoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))\n        self.padder_size = 2 ** len(enc_blks)\n\n    def forward(self, inp):\n        B, C, H, W = inp.shape\n        ph = (self.padder_size - H % self.padder_size) % self.padder_size\n        pw = (self.padder_size - W % self.padder_size) % self.padder_size\n        x = F.pad(inp, (0, pw, 0, ph), 'reflect') if ph or pw else inp\n        res = x\n        x = self.intro(x)\n        encs = []\n        for enc, down in zip(self.encoders, self.downs):\n            x = enc(x); encs.append(x); x = down(x)\n        x = self.middle_blks(x)\n        for dec, up, skip in zip(self.decoders, self.ups, reversed(encs)):\n            x = up(x) + skip; x = dec(x)\n        x = self.ending(x) + res\n        return torch.clamp(x[:, :, :H, :W] if ph or pw else x, 0., 1.)\n\ndef build_denoising_model(width=64):\n    m = NAFNet(img_channel=3, width=width,\n               enc_blks=(2,2,4,8), dec_blks=(2,2,2,2), middle_blk_num=8)\n    n = sum(p.numel() for p in m.parameters())\n    print(f'[Model] NAFNet-Ultra v10 | Width={width} | Params={n/1e6:.2f}M')\n    return m\n\n'''

with open('scripts/models.py', 'w') as f:
    f.write(models_code)
print('\u2705 scripts/models.py written successfully!')

In [ ]:
# [4] Write Dataset — scripts/dataset.py (Kaggle-compatible)
dataset_code = '''
import os, glob, random\nimport numpy as np, cv2, torch\nfrom torch.utils.data import Dataset\n\nclass DenoisingPairDataset(Dataset):\n    def __init__(self, noisy_dir, gt_dir, patch_size=256, augment=True, mixup_alpha=0.3):\n        self.noisy_dir   = noisy_dir\n        self.patch_size  = patch_size\n        self.augment     = augment\n        self.mixup_alpha = mixup_alpha\n        self.gt_files    = sorted(glob.glob(os.path.join(gt_dir, '*.png')))\n        if not self.gt_files:\n            raise ValueError(f'No GT images in {gt_dir}')\n        print(f'[Dataset] Loaded {len(self.gt_files)} full training images (no holdout)')\n\n    def __len__(self): return len(self.gt_files)\n\n    def _load_pair(self, idx):\n        gp  = self.gt_files[idx]\n        bid = os.path.splitext(os.path.basename(gp))[0]\n        np_ = os.path.join(self.noisy_dir, f'{bid}_noise.png')\n        if not os.path.exists(np_):\n            np_ = os.path.join(self.noisy_dir, f'{bid}.png')\n        g = cv2.cvtColor(cv2.imread(gp),  cv2.COLOR_BGR2RGB)\n        n = cv2.cvtColor(cv2.imread(np_), cv2.COLOR_BGR2RGB)\n        return n, g\n\n    def _crop_and_aug(self, n, g):\n        h, w = n.shape[:2]\n        p = self.patch_size\n        y = random.randint(0, max(0, h-p))\n        x = random.randint(0, max(0, w-p))\n        n, g = n[y:y+p, x:x+p], g[y:y+p, x:x+p]\n        if random.random() > 0.5: n, g = np.fliplr(n).copy(), np.fliplr(g).copy()\n        if random.random() > 0.5: n, g = np.flipud(n).copy(), np.flipud(g).copy()\n        k = random.randint(0, 3)\n        if k: n, g = np.rot90(n, k).copy(), np.rot90(g, k).copy()\n        return np.ascontiguousarray(n), np.ascontiguousarray(g)\n\n    def __getitem__(self, idx):\n        n, g = self._load_pair(idx)\n        if self.augment:\n            n, g = self._crop_and_aug(n, g)\n            if self.mixup_alpha > 0 and random.random() > 0.5:\n                idx2 = random.randint(0, len(self.gt_files)-1)\n                n2, g2 = self._load_pair(idx2)\n                n2, g2 = self._crop_and_aug(n2, g2)\n                lam = float(np.random.beta(self.mixup_alpha, self.mixup_alpha))\n                n = (lam * n.astype(np.float32) + (1-lam) * n2.astype(np.float32)).astype(np.uint8)\n                g = (lam * g.astype(np.float32) + (1-lam) * g2.astype(np.float32)).astype(np.uint8)\n        return (torch.from_numpy(n).permute(2,0,1).float()/255.,\n                torch.from_numpy(g).permute(2,0,1).float()/255.)\n\n'''

with open('scripts/dataset.py', 'w') as f:
    f.write(dataset_code)
print('\u2705 scripts/dataset.py written successfully!')

In [ ]:
# [5] Load Dataset — Kaggle & Colab Compatible
import os, glob, shutil

NOISY_DIR = 'competition_data/public/noisy'
GT_DIR    = 'competition_data/public/ground_truth'

# Try Colab Drive mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    matches = (glob.glob('/content/drive/MyDrive/**/competition_data/public', recursive=True) +
               glob.glob('/content/drive/MyDrive/**/public', recursive=True))
    if matches:
        dp = matches[0]
        print(f'\u2705 Colab Drive dataset: {dp}')
        os.system(f'cp -rn {dp}/noisy/* {NOISY_DIR}/ 2>/dev/null')
        os.system(f'cp -rn {dp}/ground_truth/* {GT_DIR}/ 2>/dev/null')
except Exception:
    pass  # Not Colab — continue

# Try Kaggle dataset paths
kaggle_candidates = [
    '/kaggle/input',
    '/kaggle/working',
]
for base in kaggle_candidates:
    found_noisy = glob.glob(f'{base}/**/noisy', recursive=True)
    found_gt    = glob.glob(f'{base}/**/ground_truth', recursive=True)
    if found_noisy:
        print(f'\u2705 Kaggle dataset noisy found: {found_noisy[0]}')
        os.system(f'cp -rn {found_noisy[0]}/* {NOISY_DIR}/ 2>/dev/null')
    if found_gt:
        print(f'\u2705 Kaggle dataset GT found: {found_gt[0]}')
        os.system(f'cp -rn {found_gt[0]}/* {GT_DIR}/ 2>/dev/null')

nn_ = len(glob.glob(f'{NOISY_DIR}/*.png'))
ng  = len(glob.glob(f'{GT_DIR}/*.png'))
print(f'\U0001f4ca Dataset: {nn_} noisy | {ng} ground truth images')
if ng == 0:
    raise RuntimeError('\u274c No ground truth found! Upload dataset to Kaggle or Drive.')

In [ ]:
# [6] ===== v10 APEX: FULL POWER TRAINING ENGINE (200 EPOCHS) =====
import sys
sys.path.insert(0, 'scripts')
from models import build_denoising_model
from dataset import DenoisingPairDataset
import torch, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR
import time, gc, os, copy, numpy as np

gc.collect(); torch.cuda.empty_cache()
device = torch.device('cuda')

# ─── HYPER-PARAMETERS ────────────────────────────────────────────
WIDTH        = 64
PATCH_SIZE   = 256
BATCH_SIZE   = 4
ACCUM_STEPS  = 4
TOTAL_EPOCHS = 200
PEAK_LR      = 6e-4
MIN_LR       = 1e-7
EMA_DECAY    = 0.999
SWA_START_EP = 181
SWA_LR       = 1e-5
PHASE2_START = 101
PHASE3_START = 161
NOISY_DIR    = 'competition_data/public/noisy'
GT_DIR       = 'competition_data/public/ground_truth'
# ─────────────────────────────────────────────────────────────────

full_ds = DenoisingPairDataset(NOISY_DIR, GT_DIR, patch_size=PATCH_SIZE,
                                augment=True, mixup_alpha=0.3)
train_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)
print(f'\U0001f680 Full Training: {len(full_ds)} images | {len(train_loader)} batches/epoch')

model = build_denoising_model(width=WIDTH).to(device)
print('[*] \U0001f195 FRESH RANDOM INIT — New loss landscape!')
print('[*] Benchmark to beat: Score=0.6205 | PSNR=30.8025 dB | SSIM=0.9179')

class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters(): p.requires_grad_(False)
        self.decay = decay
    def update(self, model):
        with torch.no_grad():
            for ep, mp in zip(self.ema_model.parameters(), model.parameters()):
                ep.data.mul_(self.decay).add_(mp.data, alpha=1.0 - self.decay)

ema = ModelEMA(model, decay=EMA_DECAY)
swa_model  = AveragedModel(model)
swa_active = False

optimizer = optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=50, T_mult=1, eta_min=MIN_LR)
swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=5)
scaler = torch.amp.GradScaler('cuda')

def charbonnier(pred, gt, eps=1e-6):
    return torch.mean(torch.sqrt((pred - gt)**2 + eps))

def multiscale_fft_loss(pred, gt, scales=[1, 2, 4]):
    total = 0.0
    for s in scales:
        p_ = F.avg_pool2d(pred, s, s) if s > 1 else pred
        g_ = F.avg_pool2d(gt,   s, s) if s > 1 else gt
        total += torch.mean(torch.abs(
            torch.fft.rfft2(p_, norm='ortho') - torch.fft.rfft2(g_, norm='ortho')))
    return total / len(scales)

def soft_ssim_loss(pred, gt):
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(pred, 11, 1, 5)
    mu_g = F.avg_pool2d(gt,   11, 1, 5)
    mu_pp, mu_gg, mu_pg = mu_p**2, mu_g**2, mu_p*mu_g
    sg_pp = F.avg_pool2d(pred*pred, 11, 1, 5) - mu_pp
    sg_gg = F.avg_pool2d(gt*gt,     11, 1, 5) - mu_gg
    sg_pg = F.avg_pool2d(pred*gt,   11, 1, 5) - mu_pg
    return 1.0 - torch.mean(
        (2*mu_pg + C1)*(2*sg_pg + C2)/((mu_pp+mu_gg+C1)*(sg_pp+sg_gg+C2)))

def get_loss(pred, gt, epoch):
    loss = charbonnier(pred, gt)
    if epoch >= PHASE2_START: loss = loss + 0.05 * multiscale_fft_loss(pred, gt)
    if epoch >= PHASE3_START: loss = loss + 0.03 * soft_ssim_loss(pred, gt)
    return loss

def get_phase(epoch):
    if epoch < PHASE2_START:   return 'P1'
    elif epoch < PHASE3_START: return 'P2'
    else:                       return 'P3'

BASE_PSNR, BASE_SSIM = 19.8257, 0.4644

def psnr_fn(pred, gt):
    return 10. * torch.log10(1. / torch.mean((pred-gt)**2).clamp(1e-8)).item()

def ssim_fn(pred, gt):
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(pred, 11, 1, 5)
    mu_g = F.avg_pool2d(gt,   11, 1, 5)
    mu_pp, mu_gg, mu_pg = mu_p**2, mu_g**2, mu_p*mu_g
    sg_pp = F.avg_pool2d(pred*pred, 11, 1, 5) - mu_pp
    sg_gg = F.avg_pool2d(gt*gt,     11, 1, 5) - mu_gg
    sg_pg = F.avg_pool2d(pred*gt,   11, 1, 5) - mu_pg
    return torch.mean((2*mu_pg+C1)*(2*sg_pg+C2)/((mu_pp+mu_gg+C1)*(sg_pp+sg_gg+C2))).item()

def composite(psnr, ssim_v):
    return 0.6*min(max((psnr-BASE_PSNR)/15., 0.), 1.) + 0.4*max(ssim_v-BASE_SSIM, 0.)

def quick_eval(net, loader, n_batches=3):
    net.eval()
    p_list, s_list = [], []
    with torch.no_grad():
        for i, (nv, gv) in enumerate(loader):
            if i >= n_batches: break
            nv, gv = nv.to(device), gv.to(device)
            with torch.amp.autocast('cuda'):
                pv = net(nv)
            p_list.append(psnr_fn(pv, gv))
            s_list.append(ssim_fn(pv, gv))
    vp = sum(p_list)/len(p_list)
    vs = sum(s_list)/len(s_list)
    return vp, vs, composite(vp, vs)

CKPT_V10 = 'scripts/checkpoints/best_denoiser_v10.pth'
os.makedirs('scripts/checkpoints', exist_ok=True)

# Kaggle/Colab Drive checkpoint paths
DRIVE_PATHS = [
    '/content/drive/MyDrive/best_denoiser_v10.pth',
    '/kaggle/working/best_denoiser_v10.pth',
]

def save_checkpoint(state_dict, width, psnr, ssim, score, epoch, tag):
    ckpt = {'model_state_dict': state_dict, 'width': width,
            'psnr': psnr, 'ssim': ssim, 'composite_score': score,
            'epoch': epoch, 'type': tag}
    torch.save(ckpt, CKPT_V10)
    for dp in DRIVE_PATHS:
        try:
            if os.path.exists(os.path.dirname(dp)):
                torch.save(ckpt, dp)
        except Exception:
            pass

best_score, best_psnr, best_ssim = 0.0, 0.0, 0.0

print('=' * 90)
print(f'v10 APEX | Ep: {TOTAL_EPOCHS} | LR: {PEAK_LR} | WarmRestarts T0=50 | Full {len(full_ds)}-img train')
print(f'Phase 1: Ep 1-100 | Phase 2: Ep 101-160 | Phase 3: Ep 161-200 | SWA: Ep 181-200')
print('=' * 90)

for epoch in range(1, TOTAL_EPOCHS + 1):
    if epoch in [PHASE2_START, PHASE3_START]:
        print(f'\n\U0001f4c8 LOSS UPGRADE → {get_phase(epoch)}')
    if epoch == SWA_START_EP:
        print(f'\n\U0001f300 SWA ACTIVATED at epoch {epoch}!')
        swa_active = True

    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    t0 = time.time()

    for i, (noisy, gt) in enumerate(train_loader):
        noisy, gt = noisy.to(device, non_blocking=True), gt.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            pred = model(noisy)
            loss = get_loss(pred, gt, epoch) / ACCUM_STEPS
        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS
        if (i+1) % ACCUM_STEPS == 0 or (i+1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.01)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
            ema.update(model)

    if swa_active:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()

    cur_lr   = optimizer.param_groups[0]['lr']
    avg_loss = total_loss / len(train_loader)
    elapsed  = time.time() - t0
    model.eval()
    torch.cuda.empty_cache()

    vp_s, vs_s, comp_s = quick_eval(model, train_loader)
    vp_e, vs_e, comp_e = quick_eval(ema.ema_model, train_loader)
    candidates = [(comp_s, vp_s, vs_s, model.state_dict(), '[STD]'),
                  (comp_e, vp_e, vs_e, ema.ema_model.state_dict(), '[EMA]')]
    if swa_active and epoch % 5 == 0:
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)
        p_sw, s_sw, c_sw = quick_eval(swa_model, train_loader)
        candidates.append((c_sw, p_sw, s_sw, swa_model.state_dict(), '[SWA]'))

    cur_comp, cur_p, cur_s, best_w, tag = max(candidates, key=lambda x: x[0])
    flag = ''
    if cur_comp > best_score:
        best_score, best_psnr, best_ssim = cur_comp, cur_p, cur_s
        save_checkpoint(best_w, WIDTH, cur_p, cur_s, cur_comp, epoch, tag)
        flag = f'  \U0001f3af RECORD! Score={cur_comp:.4f}'

    cyc = ((epoch-1)//50)+1
    print(f'Ep [{epoch:03d}/{TOTAL_EPOCHS}] C{cyc} {get_phase(epoch)} | '
          f'Loss: {avg_loss:.4f} | PSNR: {cur_p:.2f} dB | SSIM: {cur_s:.4f} | '
          f'Score: {cur_comp:.4f} {tag} | LR: {cur_lr:.1e} | {elapsed:.0f}s{flag}')

    if epoch % 50 == 0 and epoch < TOTAL_EPOCHS:
        print(f'\n\U0001f504 COSINE CYCLE {cyc} DONE | Best: Score={best_score:.4f} PSNR={best_psnr:.2f}\n')
    torch.cuda.empty_cache()

if swa_active:
    print('\n[*] Final SWA BN update...')
    torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)
    p_f, s_f, c_f = quick_eval(swa_model, train_loader)
    if c_f > best_score:
        best_score, best_psnr, best_ssim = c_f, p_f, s_f
        save_checkpoint(swa_model.state_dict(), WIDTH, p_f, s_f, c_f, TOTAL_EPOCHS, '[SWA-FINAL]')
        print('\U0001f300 SWA FINAL IS BEST!')

print('=' * 90)
print('\U0001f3c6 v10 APEX COMPLETE!')
print(f'   Best Score: {best_score:.4f} | PSNR: {best_psnr:.4f} dB | SSIM: {best_ssim:.4f}')
if best_score >= 0.6205:
    print(f'   \U0001f3c6 BEAT v7 CHAMPION! Gain: +{best_score-0.6205:.4f}')
print('=' * 90)

In [ ]:
# [7] Verify & Download Checkpoint
import hashlib, os

CKPT_V10 = 'scripts/checkpoints/best_denoiser_v10.pth'

if os.path.exists(CKPT_V10):
    sha = hashlib.sha256()
    with open(CKPT_V10, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''): sha.update(chunk)
    print('=' * 90)
    print(f'SHA-256: {sha.hexdigest()}')
    print(f'Best Score: {best_score:.4f} | PSNR: {best_psnr:.4f} dB | SSIM: {best_ssim:.4f}')
    print('=' * 90)

    # Kaggle: copy to /kaggle/working for download
    import shutil
    kaggle_out = '/kaggle/working/best_denoiser_v10.pth'
    try:
        shutil.copy2(CKPT_V10, kaggle_out)
        print(f'\u2705 Kaggle: Model saved to {kaggle_out}')
        print('   Go to Kaggle → Output tab → Download best_denoiser_v10.pth')
    except Exception:
        pass

    # Colab: auto-download
    try:
        from google.colab import files
        print('\U0001f4e5 Colab: Downloading best_denoiser_v10.pth...')
        files.download(CKPT_V10)
    except Exception:
        pass
else:
    print('\u274c No checkpoint found — training may not have completed.')